In [2]:
!pip install lifelines
!pip install sympy==1.13.3

"""
RNN-AFT (GRU) with vectorized Gehan-type loss and sub-sampling-pairs strategy.

This file extends the previous RNN-AFT implementation by adding:
- A sub-sampling-pairs procedure that, for each uncensored residual e_{ij}, samples s
  other residuals from the pool of all residuals (excluding itself).
- A training loop that, at the start of each epoch, obtains a snapshot of current
  predictions (used only for sampling), builds m = n' * s sampled pairs, shuffles them,
  and then iterates over these pairs in mini-batches of pairs (size b). For each
  mini-batch of pairs we run a differentiable forward pass over the small subset of
  subjects involved in the pairs and compute a properly-weighted unbiased estimator
  of the Gehan loss for the mini-batch (with reduction='mean' as before).
- Helper utilities for sampling and mapping pair indices back to subject/time indices.

Notes on unbiasedness:
- Let N_total be the total number of residuals (events) across dataset (including
  censored events and all subjects), and n' be the number of uncensored residuals.
  For each uncensored residual we sample s draws from the remaining N_total-1
  residuals. The resulting sampled-sum is scaled by (N_total - 1) / s so that its
  expectation equals the full per-uncensored-event sum. With reduction='mean' we
  then divide by n' (the number of uncensored events) to match the earlier GehanLoss
  mean reduction: final scaling factor = (N_total - 1) / (n' * s).
- Implementation follows that weighting.

Author: (Your Name)
Dependencies: numpy, torch, lifelines, pandas, matplotlib
"""

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from lifelines import KaplanMeierFitter
import pandas as pd
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict

# === Model & utilities (same core model as before, forward accepts concatenated input) ===

class RNNAFTGRU(nn.Module):
    def __init__(self, cov_dim, hidden_dim=32, gru_layers=1, dropout=0.0):
        super().__init__()
        self.cov_dim = cov_dim
        self.input_dim = 1 + cov_dim
        self.hidden_dim = hidden_dim
        self.gru_layers = gru_layers

        self.gru = nn.GRU(
            input_size=self.input_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0
        )
        self.output = nn.Linear(hidden_dim, 1)
        self.init_fc = nn.Linear(cov_dim, gru_layers * hidden_dim)

    def initialize_hidden(self, X_static):
        h0_flat = torch.tanh(self.init_fc(X_static))
        h0 = h0_flat.view(self.gru_layers, X_static.size(0), self.hidden_dim)
        return h0

    def forward(self, X_in):
        # X_in: [batch, seq, 1 + cov_dim]
        covariates_seq = X_in[..., 1:]
        X_static = covariates_seq[:, 0, :]
        h0 = self.initialize_hidden(X_static)
        gru_out, h = self.gru(X_in, h0)
        yhat = self.output(gru_out).squeeze(-1)  # [batch, seq]
        return yhat, h

def batchify_subjects(subject_data: List[Dict], cov_dim: int):
    batch_size = len(subject_data)
    seq_lens = [len(s['log_gaps']) for s in subject_data]
    max_seq = max(seq_lens) if len(seq_lens) > 0 else 0
    prev_log_gaps = np.zeros((batch_size, max_seq, 1), dtype=np.float32)
    covariates = np.zeros((batch_size, max_seq, cov_dim), dtype=np.float32)
    log_gaps = np.zeros((batch_size, max_seq), dtype=np.float32)
    delta = np.zeros((batch_size, max_seq), dtype=np.float32)
    for i, s in enumerate(subject_data):
        l = seq_lens[i]
        if l == 0:
            continue
        if l > 1:
            prev_log_gaps[i, 1:l, 0] = s['log_gaps'][:-1]
        covariates[i, :l, :] = np.tile(s['covariates'], (l, 1))
        log_gaps[i, :l] = s['log_gaps']
        delta[i, :l] = s['delta']
    return (
        torch.tensor(prev_log_gaps, dtype=torch.float32),
        torch.tensor(covariates, dtype=torch.float32),
        torch.tensor(log_gaps, dtype=torch.float32),
        torch.tensor(delta, dtype=torch.float32),
        torch.tensor(seq_lens, dtype=torch.long)
    )

# === Gehan loss (full version left for reference) ===

class GehanLoss(nn.Module):
    """
    Original (full) Gehan loss that sums over all pairs. Kept for reference/testing.
    """
    def __init__(self, reduction='mean'):
        super().__init__()
        self.reduction = reduction

    def forward(self, pred_log_gaps, true_log_gaps, delta, seq_lens):
        device = pred_log_gaps.device
        batch, max_seq = pred_log_gaps.shape
        mask = (torch.arange(max_seq, device=device)[None, :] < seq_lens[:, None])
        events_mask = (delta == 1) & mask

        event_indices = events_mask.nonzero(as_tuple=False)
        if len(event_indices) == 0:
            return torch.tensor(0.0, requires_grad=True, device=device)

        e = true_log_gaps - pred_log_gaps  # residuals
        flat_e = e[mask]  # flatten only valid positions
        e_ij = e[event_indices[:, 0], event_indices[:, 1]]

        diffs = flat_e.unsqueeze(0) - e_ij.unsqueeze(1)  # [num_events, N_total]
        neg_part = torch.abs(torch.minimum(diffs, torch.zeros_like(diffs)))
        loss = neg_part.sum()
        if self.reduction == 'mean':
            loss = loss / event_indices.shape[0]
        return loss

# === Sub-sampling pairs utilities ===

def flatten_events(subjects: List[Dict]) -> Tuple[int, np.ndarray, np.ndarray, np.ndarray]:
    """
    Flatten events and provide mapping arrays.

    Returns:
      N_total: int (total number of event slots across subjects)
      subj_of_flat: np.array shape (N_total,) mapping flat_idx -> subject index i
      t_of_flat: np.array shape (N_total,) mapping flat_idx -> event time index j
      is_valid: np.array shape (N_total,) boolean - True if event exists (no padding)
    """
    seq_lens = [len(s['log_gaps']) for s in subjects]
    N_total = sum(seq_lens)
    subj_of_flat = np.empty(N_total, dtype=np.int32)
    t_of_flat = np.empty(N_total, dtype=np.int32)
    pos = 0
    for i, L in enumerate(seq_lens):
        for j in range(L):
            subj_of_flat[pos] = i
            t_of_flat[pos] = j
            pos += 1
    return N_total, subj_of_flat, t_of_flat

def sample_pairs_from_predictions(
    subjects: List[Dict],
    pred_log: np.ndarray,
    s: int,
    seed: int = None
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    For each uncensored residual (i,j) sample s indices from all other residuals
    (flat indexing). Returns arrays (I, J, L, K) each length m = n' * s, where
      (I[t], J[t]) is the left element (uncensored event),
      (L[t], K[t]) is the sampled counterpart.
    Sampling is uniform over other residuals excluding the same index.
    """
    rng = np.random.default_rng(seed)
    # Build flatten mapping and helpful arrays
    seq_lens = [len(su['log_gaps']) for su in subjects]
    prefix = np.concatenate([[0], np.cumsum(seq_lens)])
    N_total = int(prefix[-1])
    if N_total == 0:
        return (np.array([], dtype=int),) * 4

    # list of all flat indices
    all_flat = np.arange(N_total, dtype=int)

    # map from flat index to (i,j)
    subj_of_flat = np.empty(N_total, dtype=int)
    t_of_flat = np.empty(N_total, dtype=int)
    pos = 0
    for i, L in enumerate(seq_lens):
        for j in range(L):
            subj_of_flat[pos] = i
            t_of_flat[pos] = j
            pos += 1

    # gather uncensored flat indices
    uncensored_flats = []
    for i, ssub in enumerate(subjects):
        for j in range(len(ssub['log_gaps'])):
            if int(ssub['delta'][j]) == 1:
                flat_idx = prefix[i] + j
                uncensored_flats.append(flat_idx)
    uncensored_flats = np.array(uncensored_flats, dtype=int)
    n_prime = len(uncensored_flats)
    if n_prime == 0:
        return (np.array([], dtype=int),) * 4

    m = n_prime * s
    I = np.empty(m, dtype=int)
    J = np.empty(m, dtype=int)
    L = np.empty(m, dtype=int)
    K = np.empty(m, dtype=int)
    out_pos = 0
    for flat in uncensored_flats:
        # candidate pool excludes self
        pool = np.delete(all_flat, flat)
        replace = False if (pool.size >= s) else True
        chosen = rng.choice(pool, size=s, replace=replace)
        for ch in chosen:
            Ii = subj_of_flat[flat]
            Ji = t_of_flat[flat]
            Ll = subj_of_flat[ch]
            Kl = t_of_flat[ch]
            I[out_pos] = Ii
            J[out_pos] = Ji
            L[out_pos] = Ll
            K[out_pos] = Kl
            out_pos += 1
    return I, J, L, K

# === Metrics (unchanged from previous corrected implementation) ===

def pad_log_gaps(subjects, max_seq):
    n = len(subjects)
    arr = np.zeros((n, max_seq), dtype=np.float32)
    for i, s in enumerate(subjects):
        l = len(s['log_gaps'])
        arr[i, :l] = s['log_gaps']
    return arr

def compute_mse(subjects, pred_log):
    rows = []
    for i, subj in enumerate(subjects):
        for j in range(len(subj['log_gaps'])):
            rows.append({
                'i': i,
                'j': j,
                'log_gap': float(subj['log_gaps'][j]),
                'gap': float(np.exp(subj['log_gaps'][j])),
                'delta': int(subj['delta'][j]),
                'pred_log': float(pred_log[i, j])
            })
    df = pd.DataFrame(rows)
    if df.shape[0] == 0:
        return np.nan
    kmf = KaplanMeierFitter()
    try:
        kmf.fit(df['gap'], event_observed=1 - df['delta'])
        df['G_hat'] = kmf.predict(df['gap']).values
    except Exception:
        df['G_hat'] = 1.0
    df['G_hat'] = np.clip(df['G_hat'], 1e-6, 1.0)
    se = df['delta'] / df['G_hat'] * (df['log_gap'] - df['pred_log'])**2
    denom = df['delta'].sum()
    if denom == 0:
        return np.nan
    amse = se.sum() / denom
    return float(amse)

def ipcw_cindex(subjects, pred_log):
    rows = []
    for i, subj_i in enumerate(subjects):
        for j in range(len(subj_i['log_gaps'])):
            rows.append({
                'i': i,
                'j': j,
                'gap': float(np.exp(subj_i['log_gaps'][j])),
                'log_gap': float(subj_i['log_gaps'][j]),
                'delta': int(subj_i['delta'][j]),
                'pred_log': float(pred_log[i, j])
            })
    df = pd.DataFrame(rows)
    if df.shape[0] == 0:
        return np.nan
    kmf = KaplanMeierFitter()
    try:
        kmf.fit(df['gap'], event_observed=1 - df['delta'])
        df['G_hat'] = kmf.predict(df['gap']).values
    except Exception:
        df['G_hat'] = 1.0
    df['G_hat'] = np.clip(df['G_hat'], 1e-6, 1.0)
    num, denom = 0.0, 0.0
    for idx_i, row_i in df.iterrows():
        if row_i['delta'] != 1:
            continue
        for idx_k, row_k in df.iterrows():
            if row_i['i'] == row_k['i']:
                continue
            if row_i['gap'] < row_k['gap']:
                weight = 1.0 / row_i['G_hat'] if row_i['G_hat'] > 0 else 0.0
                concordant = 1.0 if row_i['pred_log'] < row_k['pred_log'] else 0.0
                num += weight * concordant
                denom += weight
    return float(num / denom) if denom > 0 else np.nan

# === Training with sub-sampled pairs ===

def train_rnn_aft_subsample_pairs(
    train_subjects,
    test_subjects,
    cov_dim=3,
    hidden_dim=32,
    epochs=10,
    pair_sample_s=2,
    pair_batch_b=64,
    lr=1e-3,
    device="cpu",
    seed=None
):
    device = torch.device(device)
    rng = np.random.default_rng(seed)
    model = RNNAFTGRU(cov_dim, hidden_dim=hidden_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n_train = len(train_subjects)
    train_losses = []
    train_mse_hist = []
    test_mse_hist = []
    train_cidx_hist = []
    test_cidx_hist = []

    def predict_all_numpy(subjects):
        if len(subjects) == 0:
            return np.zeros((0, 0), dtype=np.float32)
        n = len(subjects)
        max_seq = max([len(s['log_gaps']) for s in subjects])
        X_prev_gaps, X_cov, Y_log_gaps, Y_delta, seq_lens = batchify_subjects(subjects, cov_dim)
        X_in = torch.cat([X_prev_gaps, X_cov], dim=-1).to(device)
        with torch.no_grad():
            yhat, _ = model(X_in)
        pred_log = yhat.cpu().numpy()
        out = np.zeros_like(pred_log)
        for i in range(n):
            out[i, :seq_lens[i]] = pred_log[i, :seq_lens[i]]
        return out

    for epoch in range(epochs):
        model.train()
        # Step 1: snapshot predictions to sample pairs (use numpy snapshot)
        pred_log_train_snapshot = predict_all_numpy(train_subjects)

        # Step 2: sample pairs (m = n' * s)
        I, J, L, K = sample_pairs_from_predictions(train_subjects, pred_log_train_snapshot, pair_sample_s, seed=rng.integers(1_000_000))
        m = I.size
        # If nothing was sampled (no uncensored events), fall back to no-op update pass
        if m == 0:
            print(f"Epoch {epoch+1}: no uncensored events found, skipping pair-subsampling updates.")
            train_losses.append(0.0)
            train_mse_hist.append(compute_mse(train_subjects, pred_log_train_snapshot))
            test_mse_hist.append(compute_mse(test_subjects, predict_all_numpy(test_subjects)))
            train_cidx_hist.append(ipcw_cindex(train_subjects, pred_log_train_snapshot))
            test_cidx_hist.append(ipcw_cindex(test_subjects, predict_all_numpy(test_subjects)))
            continue

        # shuffle sampled pairs
        perm_pairs = rng.permutation(m)
        I, J, L, K = I[perm_pairs], J[perm_pairs], L[perm_pairs], K[perm_pairs]

        # Precompute some constants for weighting
        N_total = sum(len(s['log_gaps']) for s in train_subjects)
        # number of uncensored events n'
        n_prime = int(np.sum([np.sum(s['delta']) for s in train_subjects]))

        # per-epoch optimization over sampled pairs mini-batches
        losses_epoch = []
        # iterate mini-batches of pairs
        for start in range(0, m, pair_batch_b):
            end = min(start + pair_batch_b, m)
            batch_slice = slice(start, end)
            Ib, Jb, Lb, Kb = I[batch_slice], J[batch_slice], L[batch_slice], K[batch_slice]

            # find unique subject indices in this pairs-batch
            unique_subjects, inv_idx = np.unique(np.concatenate([Ib, Lb]), return_inverse=True)
            # map global subject idx -> local index
            g2l = {int(g): int(l) for l, g in enumerate(unique_subjects)}
            # build small sub-batch of subjects
            sub_subjects = [train_subjects[int(g)] for g in unique_subjects]
            X_prev_gaps, X_cov, Y_log_gaps, Y_delta, seq_lens = batchify_subjects(sub_subjects, cov_dim)
            X_in = torch.cat([X_prev_gaps, X_cov], dim=-1).to(device)
            Y_log_gaps = Y_log_gaps.to(device)
            Y_delta = Y_delta.to(device)
            seq_lens_t = seq_lens.to(device)

            optimizer.zero_grad()
            yhat_sub, _ = model(X_in)  # [n_sub, max_sub_seq]
            # For each pair we need predicted log for left and right
            # Build tensors of predictions corresponding to pairs
            left_preds = []
            right_preds = []
            left_trues = []
            right_trues = []
            for idx_pair in range(Ib.size):
                gi = int(Ib[idx_pair]); gj = int(Jb[idx_pair])
                gl = int(Lb[idx_pair]); gk = int(Kb[idx_pair])
                li = g2l[gi]; ll = g2l[gl]
                # grab as torch scalars
                # Note: ensure event index within seq_lens for sub-batch (should be by construction)
                pred_ij = yhat_sub[li, gj]
                pred_lk = yhat_sub[ll, gk]
                left_preds.append(pred_ij)
                right_preds.append(pred_lk)
                left_trues.append(torch.tensor(float(train_subjects[gi]['log_gaps'][gj]), device=device))
                right_trues.append(torch.tensor(float(train_subjects[gl]['log_gaps'][gk]), device=device))
            left_preds = torch.stack(left_preds)  # [batch_pairs_in_minibatch]
            right_preds = torch.stack(right_preds)
            left_trues = torch.stack(left_trues)
            right_trues = torch.stack(right_trues)

            # residuals e = true - pred
            e_ij = left_trues - left_preds
            e_lk = right_trues - right_preds
            diffs = e_lk - e_ij
            neg_part = torch.abs(torch.minimum(diffs, torch.zeros_like(diffs)))
            sum_neg = neg_part.sum()

            # scaling for unbiased estimator:
            # For each uncensored event we sampled s from N_total - 1 possibilities. So factor = (N_total - 1) / s
            # Our overall GehanLoss with reduction='mean' divides by n' (#uncensored events).
            scale = float((N_total - 1) / max(1, pair_sample_s))
            loss_batch = sum_neg * scale
            if n_prime > 0:
                loss_batch = loss_batch / float(n_prime)
            # backpropagate
            loss_batch.backward()
            optimizer.step()
            losses_epoch.append(float(loss_batch.detach().cpu().numpy()))

        mean_epoch_loss = float(np.mean(losses_epoch)) if len(losses_epoch) > 0 else 0.0
        train_losses.append(mean_epoch_loss)

        # Evaluate on train and test sets (use current model to predict)
        model.eval()
        pred_log_train = predict_all_numpy(train_subjects)
        pred_log_test = predict_all_numpy(test_subjects)
        mse_train = compute_mse(train_subjects, pred_log_train)
        mse_test = compute_mse(test_subjects, predict_all_numpy(test_subjects))
        train_mse_hist.append(mse_train)
        test_mse_hist.append(mse_test)
        cidx_train = ipcw_cindex(train_subjects, pred_log_train)
        cidx_test = ipcw_cindex(test_subjects, predict_all_numpy(test_subjects))
        train_cidx_hist.append(cidx_train)
        test_cidx_hist.append(cidx_test)

        print(f"Epoch {epoch + 1}/{epochs} | SamplePairs m={m} | Mean Pair-Batch Loss: {mean_epoch_loss:.4f} | Train MSE: {mse_train:.4f} | Test MSE: {mse_test:.4f} | Train C-index: {np.nan_to_num(cidx_train):.3f} | Test C-index: {np.nan_to_num(cidx_test):.3f}")

    # final evaluation
    model.eval()
    pred_log_train = predict_all_numpy(train_subjects)
    pred_log_test = predict_all_numpy(test_subjects)
    mse_train = compute_mse(train_subjects, pred_log_train)
    mse_test = compute_mse(test_subjects, pred_log_test)
    cidx_train = ipcw_cindex(train_subjects, pred_log_train)
    cidx_test = ipcw_cindex(test_subjects, pred_log_test)

    return {
        "model": model,
        "mse_train": mse_train,
        "mse_test": mse_test,
        "cindex_train": cidx_train,
        "cindex_test": cidx_test,
        "pred_log_train": pred_log_train,
        "pred_log_test": pred_log_test,
        "train_losses": train_losses,
        "train_mse_hist": train_mse_hist,
        "test_mse_hist": test_mse_hist,
        "train_cidx_hist": train_cidx_hist,
        "test_cidx_hist": test_cidx_hist,
    }

# === Small smoke test runner (kept minimal) ===

def generate_covariates(n):
    x1 = np.random.binomial(1, 0.5, size=n)
    x2 = np.random.normal(loc=x1 / 3, scale=1, size=n)
    x3 = np.random.normal(loc=x2 / 3, scale=1, size=n)
    return np.column_stack([x1, x2, x3]).astype(np.float32)

def f1_nonlinear(x):
    return 3 * x[:, 0] + 7 * x[:, 0] * x[:, 1] - 5 * x[:, 2]

def f_gam(X):
    """
    GAM-type nonlinear mean function:
    f(X) = x1 + 2*x2^3 + sin(0.9*x3)
    """
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    return x1 + (x2 ** 3) + np.exp(0.9 * x3)

def f_linear(X):
    """
    Linear mean function:
    f(X) = x1 - 6*x2 + 4*x3
    """
    x1, x2, x3 = X[:, 0], X[:, 1], X[:, 2]
    return x1 - 6 * x2 + 4 * x3

def get_error(name, size):
    """
    Generate error draws.

    Supported options:
      - "normal"  : standard normal N(0,1)
      - "gumbel"  : standard Gumbel with loc=0, scale=1
      - "logistic": standard Logistic with loc=0, scale=1

    Returns a NumPy array of length `size`.
    """
    if name == "normal":
        # standard normal (mean 0, std 1)
        return np.random.normal(0, 1, size)
    elif name == "gumbel":
        # standard Gumbel (loc=0, scale=1)
        return np.random.gumbel(0, 1, size)
    elif name == "logistic":
        # standard logistic (loc=0, scale=1)
        return np.random.logistic(0, 1, size)
    else:
        raise ValueError("Unknown error dist")

def generate_gap_times(n, mean_func, error_dist, sigma_b=0.5, ar1_rho=None, poisson_lambda=2):
    X = generate_covariates(n)
    means = mean_func(X)
    K = np.random.poisson(lam=poisson_lambda, size=n)
    subject_list = []
    for i in range(n):
        ki = K[i] if K[i] > 0 else 1
        xi = X[i]
        mui = means[i]
        bi = np.random.normal(0, sigma_b)
        if ar1_rho is not None:
            eta = np.zeros(ki)
            eta[0] = get_error(error_dist, 1)[0]
            for j in range(1, ki):
                eta[j] = ar1_rho * eta[j - 1] + np.sqrt(1 - ar1_rho ** 2) * get_error(error_dist, 1)[0]
        else:
            eta = get_error(error_dist, ki)
        eps = bi + eta
        log_gaps = mui + eps
        gaps = np.exp(log_gaps)
        subject_list.append({
            "covariates": xi.astype(np.float32),
            "log_gaps": log_gaps.astype(np.float32),
            "gaps": gaps.astype(np.float32),
            "frailty": float(bi)
        })
    return subject_list

def apply_censoring(subject_list, tau, seed=None):
    """
    Apply subject-level right-censoring consistent with the LaTeX model.

    For each subject i:
      - draw C_i ~ Uniform(0, tau) (change as needed)
      - compute cumulative sums S_j = sum_{l=1}^j T_il
      - let K_obs = max{ j : S_j <= C_i } (possibly 0)
      - for j = 1..K_obs  -> observed (delta=1), G_ij = T_ij
      - if K_obs < K_true:
            observed partial interval j = K_obs + 1 with G = C_i - S_{K_obs}, delta = 0
        else
            all gaps observed (no censoring within sequence)
    The function stores per-subject fields 'censored_gaps' and 'delta' with the same length as
    the original gaps array; entries after the censoring point remain set to zero (not observed).
    """
    import numpy as np
    rng = np.random.default_rng(seed)
    for subj in subject_list:
        gaps = np.asarray(subj['gaps'], dtype=float)   # T_{i1},...,T_{iK}
        Ki = len(gaps)
        if Ki == 0:
            subj['censored_gaps'] = np.array([], dtype=float)
            subj['delta'] = np.array([], dtype=np.int32)
            subj['C'] = np.nan
            subj['K_obs'] = 0
            continue

        # draw a subject-level censoring time C_i in (0, tau)
        C_i = float(rng.uniform(0, tau))

        # cumulative sums S_j = sum_{l=1}^j T_il
        cum = np.cumsum(gaps)

        # K_obs = number of full gaps completely observed (largest j with S_j <= C_i)
        # searchsorted with side='right' gives count of S_j <= C_i
        K_obs = int(np.searchsorted(cum, C_i, side='right'))

        # prepare outputs same length as original gaps
        censored = np.zeros_like(gaps, dtype=float)
        delta = np.zeros(Ki, dtype=np.int32)

        if K_obs >= Ki:
            # no censoring within the K gaps: all observed
            censored[:] = gaps
            delta[:] = 1
        else:
            # j = 1..K_obs observed fully
            if K_obs > 0:
                censored[:K_obs] = gaps[:K_obs]
                delta[:K_obs] = 1
            # partial last observed gap (could be at index 0 if K_obs == 0)
            prev_sum = float(cum[K_obs - 1]) if K_obs > 0 else 0.0
            partial = C_i - prev_sum
            # partial should be >= 0 by construction; store it at index K_obs (0-based)
            censored[K_obs] = float(partial)
            delta[K_obs] = 0  # last partial interval is censored indicator = 0
            # positions K_obs+1 ... Ki-1 remain zeros (not observed)

        subj['censored_gaps'] = censored
        subj['delta'] = delta
        subj['C'] = C_i
        subj['K_obs'] = K_obs
    return subject_list

def compute_censoring_proportion(subjects):
    """
    subjects: list of dicts with key 'delta' (1 = observed event, 0 = censored)
    Returns: (prop_observed, prop_censored, total_events, observed_count, censored_count)
    """
    total = 0
    observed = 0
    for s in subjects:
        if 'delta' in s:
            total += len(s['delta'])
            observed += int(np.sum(s['delta']))
    censored = total - observed
    prop_observed = observed / total if total > 0 else float('nan')
    prop_censored = censored / total if total > 0 else float('nan')
    return prop_observed, prop_censored, total, observed, censored


def prepare_subjects_for_nn(subject_list):
    subjects = []
    for subj in subject_list:
        subjects.append({
            'covariates': subj['covariates'],
            'log_gaps': subj['log_gaps'],
            'delta': subj['delta']
        })
    return subjects

if __name__ == "__main__":
    # small smoke test (reduced sizes)
    np.random.seed(42)
    torch.manual_seed(42)
    mean_func = f1_nonlinear
    error_dist = "normal"
    sigma_b = 0.5
    tau = 3000
    n_train, n_test = 5000, 2000
    epochs = 3
    s = 15
    b = 128

    train_gaps = generate_gap_times(n_train, mean_func, error_dist, sigma_b)
    train_gaps = apply_censoring(train_gaps, tau)
    train_subjects = prepare_subjects_for_nn(train_gaps)

    test_gaps = generate_gap_times(n_test, mean_func, error_dist, sigma_b)
    test_gaps = apply_censoring(test_gaps, tau)
    test_subjects = prepare_subjects_for_nn(test_gaps)

    # --- NEW: print censoring / observed proportions for a given tau ---
    prop_obs_train, prop_cens_train, total_train, obs_train_cnt, cens_train_cnt = compute_censoring_proportion(train_subjects)
    prop_obs_test, prop_cens_test, total_test, obs_test_cnt, cens_test_cnt = compute_censoring_proportion(test_subjects)

    print(f"Train events: total={total_train}, observed={obs_train_cnt}, censored={cens_train_cnt}")
    print(f"Train: observed fraction (uncensored) = {prop_obs_train:.4f}, censored fraction = {prop_cens_train:.4f}")
    print(f"Test  events: total={total_test}, observed={obs_test_cnt}, censored={cens_test_cnt}")
    print(f"Test:  observed fraction (uncensored) = {prop_obs_test:.4f}, censored fraction = {prop_cens_test:.4f}")
    # ----------------------------------------------------------------

    results = train_rnn_aft_subsample_pairs(
        train_subjects, test_subjects,
        cov_dim=3, hidden_dim=32, epochs=epochs,
        pair_sample_s=s, pair_batch_b=b, lr=1e-4, device="cpu", seed=42
    )

    print("Final Train MSE:", results["mse_train"])
    print("Final Test MSE:", results["mse_test"])
    print("Final Train IPCW C-index:", results["cindex_train"])
    print("Final Test IPCW C-index:", results["cindex_test"])


Train events: total=10733, observed=7894, censored=2839
Train: observed fraction (uncensored) = 0.7355, censored fraction = 0.2645
Test  events: total=4256, observed=3197, censored=1059
Test:  observed fraction (uncensored) = 0.7512, censored fraction = 0.2488
Epoch 1/3 | SamplePairs m=118410 | Mean Pair-Batch Loss: 18.5321 | Train MSE: 9.9046 | Test MSE: 8.9075 | Train C-index: 0.849 | Test C-index: 0.844
Epoch 2/3 | SamplePairs m=118410 | Mean Pair-Batch Loss: 12.3971 | Train MSE: 7.2179 | Test MSE: 6.4503 | Train C-index: 0.886 | Test C-index: 0.882
Epoch 3/3 | SamplePairs m=118410 | Mean Pair-Batch Loss: 10.3094 | Train MSE: 6.7746 | Test MSE: 6.2740 | Train C-index: 0.908 | Test C-index: 0.904
Final Train MSE: 6.774554242352884
Final Test MSE: 6.274030588957665
Final Train IPCW C-index: 0.907711496906791
Final Test IPCW C-index: 0.9043043523753644
